# Week 46, Day 4: Tree problems II

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w046d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def split_point(node, a, b):
    while node is not None:
        if a < node.val and b < node.val:
            node = node.left
        elif a > node.val and b > node.val:
            node = node.right
        else:
            return node.val

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
print(split_point(root, 20, 35), split_point(root, 35, 40), split_point(root, 60, 90), split_point(root, 20, 90))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def lca(node, a, b):
    while True:
        if a < node.val and b < node.val:
            node = node.left
        elif a > node.val and b > node.val:
            node = node.right
        else:
            return node

def path_down(node, target):
    path = []
    while node.val != target:
        path.append(node.val)
        node = node.left if target < node.val else node.right
    return path + [target]

def route(root, a, b):
    top = lca(root, a, b)
    up = path_down(top, a)[::-1]          # from a up to the LCA
    down = path_down(top, b)[1:]          # from below the LCA down to b
    return up + down

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
for a, b in [(35, 20), (35, 90), (60, 70)]:
    r = route(root, a, b)
    print(a, "to", b, ":", r, "distance", len(r) - 1)

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def trim(node, lo, hi):
    if node is None:
        return None
    if node.val < lo:
        return trim(node.right, lo, hi)
    if node.val > hi:
        return trim(node.left, lo, hi)
    node.left = trim(node.left, lo, hi)
    node.right = trim(node.right, lo, hi)
    return node

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
print(preorder(trim(root, 33, 75)))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def greater_sum(root):
    total, stack, node = 0, [], root
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.right                 # right first: biggest values first
        node = stack.pop()
        total += node.val
        node.val = total
        node = node.left
    return root

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

T = TreeNode
print(inorder(greater_sum(T(4, T(2, T(1), T(3)), T(6, T(5), T(7))))))

In [ ]:
def find_swapped(seq):
    first = second = None
    for prev, cur in zip(seq, seq[1:]):
        if cur < prev:
            if first is None:
                first, second = prev, cur      # covers the one-inversion case
            else:
                second = cur                   # the second inversion's second value
    return first, second

print(find_swapped([1, 2, 4, 3, 5]), find_swapped([1, 5, 3, 4, 2, 6]), find_swapped([1, 2, 3]))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class InOrder:
    def __init__(self, root):
        self._stack = []
        self._push(root)

    def _push(self, node):
        while node is not None:
            self._stack.append(node)
            node = node.left

    def __iter__(self):
        return self

    def __next__(self):
        if not self._stack:
            raise StopIteration
        node = self._stack.pop()
        self._push(node.right)
        return node.val

def merge(a, b):
    ia, ib = InOrder(a), InOrder(b)
    x, y, out = next(ia, None), next(ib, None), []
    while x is not None or y is not None:
        if y is None or (x is not None and x <= y):
            out.append(x)
            x = next(ia, None)
        else:
            out.append(y)
            y = next(ib, None)
    return out

T = TreeNode
print(merge(T(5, T(1), T(9)), T(4, T(2), T(7, None, T(8)))))

In [ ]:
def count_bsts(n):
    count = [1] + [0] * n
    for size in range(1, n + 1):
        count[size] = sum(count[r - 1] * count[size - r] for r in range(1, size + 1))
    return count[n]

print([count_bsts(n) for n in range(10)])
print(count_bsts(19))

> **Key idea.** In a BST, the lowest common ancestor of two values is where their search paths split: O(h). Distances and routes follow from it. Trim a BST with the subtree-returning recursion: a node below the range is replaced by its trimmed right subtree, one above by its trimmed left. A reverse in-order walk visits values largest first, for running totals. Two swapped values show up as one or two inversions in the in-order sequence: take the first value of the first inversion and the second value of the last. Merge two BSTs with two in-order iterators in O(m + n). The number of BSTs on n values is the nth Catalan number.

### Exercise: Common ancestors and distances

Write `lca(root, a, b)` returning the LCA's **value**, or `None` if either value is missing from the tree (check both first), and `distance(root, a, b)`: the number of edges between them, or `None` if either is missing. O(h) each, with loops.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def lca(root, a, b):
    ___

def distance(root, a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_lca():\n    \"ancestors\"\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    assert [lca(root, a, b) for a, b in [(20, 35), (35, 40), (60, 90), (20, 90), (90, 60), (50, 50)]] == [30, 40, 70, 50, 70, 50]\n    assert lca(root, 20, 21) is None and lca(None, 1, 1) is None\n    assert [distance(root, a, b) for a, b in [(35, 20), (35, 90), (60, 70), (50, 50), (90, 50)]] == [3, 6, 1, 0, 3]\n    assert distance(root, 1, 50) is None")

### Exercise: Trimming to a range

Write `trim(node, lo, hi)`, returning the root of the trimmed tree, which keeps exactly the values in `lo..hi` inclusive and uses the original nodes.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def trim(node, lo, hi):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef ins(n, v):\n    if n is None:\n        return TreeNode(v)\n    if v < n.val:\n        n.left = ins(n.left, v)\n    else:\n        n.right = ins(n.right, v)\n    return n\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef test_trim():\n    \"trimming\"\n    T = TreeNode\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    keep = root.left.right\n    r = trim(root, 33, 75)\n    assert inorder(r) == [35, 40, 50, 60, 70] and r is root and r.left is keep\n    assert trim(T(5), 6, 9) is None and inorder(trim(T(5, T(1), T(9)), 2, 9)) == [5, 9]\n    rng = random.Random(12)\n    for _ in range(50):\n        vals = rng.sample(range(100), 30)\n        root = None\n        for v in vals:\n            root = ins(root, v)\n        lo, hi = sorted(rng.sample(range(100), 2))\n        assert inorder(trim(root, lo, hi)) == sorted(v for v in vals if lo <= v <= hi)")

### Exercise: Recovering a swapped pair

Write `recover(root)`: two values in this BST have been swapped; find the two **nodes** with an in-order walk using an explicit stack (no list of all values), swap their values back, and return the two values that were out of place as `(larger, smaller)`. If the tree is already a valid BST, change nothing and return `None`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def recover(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef balanced(vals, lo, hi):\n    if lo > hi:\n        return None\n    m = (lo + hi) // 2\n    return TreeNode(vals[m], balanced(vals, lo, m - 1), balanced(vals, m + 1, hi))\ndef nodes(n):\n    return [] if n is None else nodes(n.left) + [n] + nodes(n.right)\ndef test_recover():\n    \"repairs\"\n    T = TreeNode\n    r = T(3, T(1), T(4, T(2)))\n    assert recover(r) == (3, 2) and inorder(r) == [1, 2, 3, 4]\n    r = T(2, T(3), T(1))\n    assert recover(r) == (3, 1) and inorder(r) == [1, 2, 3]\n    assert recover(T(2, T(1), T(3))) is None and recover(None) is None\n    rng = random.Random(13)\n    for _ in range(200):\n        n = rng.randint(2, 40)\n        vals = sorted(rng.sample(range(1000), n))\n        root = balanced(vals, 0, n - 1)\n        a, b = rng.sample(nodes(root), 2)\n        a.val, b.val = b.val, a.val\n        got = recover(root)\n        assert inorder(root) == vals and got == (max(a.val, b.val), min(a.val, b.val))")

### Exercise: Merging two BSTs

Write `merge_bsts(a, b)`: a **balanced** BST holding every value from both trees (keeping repeats once), built in O(m + n). Merge with two in-order iterators (write the class), then build the tree from the middle outwards, as on Day 1.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class InOrder:
    ___

def merge_bsts(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, math\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef height(n):\n    return -1 if n is None else 1 + max(height(n.left), height(n.right))\ndef ins(n, v):\n    if n is None:\n        return TreeNode(v)\n    if v < n.val:\n        n.left = ins(n.left, v)\n    elif v > n.val:\n        n.right = ins(n.right, v)\n    return n\ndef test_merge():\n    \"merging\"\n    rng = random.Random(14)\n    for _ in range(30):\n        xs, ys = rng.sample(range(500), rng.randint(0, 60)), rng.sample(range(500), rng.randint(0, 60))\n        a = b = None\n        for v in xs:\n            a = ins(a, v)\n        for v in ys:\n            b = ins(b, v)\n        m = merge_bsts(a, b)\n        want = sorted(set(xs) | set(ys))\n        assert inorder(m) == want\n        assert height(m) == (math.floor(math.log2(len(want))) if want else -1)")

### Exercise: Every BST

Write `count_bsts(n)` (a bottom-up table), and `all_bsts(lo, hi)`: a list of every BST holding exactly the values `lo..hi`, each a new tree of `TreeNode`s. For each root `r`, combine every left tree from `all_bsts(lo, r - 1)` with every right tree from `all_bsts(r + 1, hi)`; an empty range gives `[None]`. (Sharing subtrees between results is fine.)

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def count_bsts(n):
    ___

def all_bsts(lo, hi):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def pre(n):\n    return () if n is None else (n.val, pre(n.left), pre(n.right))\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef test_shapes():\n    \"every shape\"\n    assert [count_bsts(n) for n in range(10)] == [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862] and count_bsts(19) == 1767263190\n    assert all_bsts(1, 0) == [None]\n    for n in range(1, 7):\n        trees = all_bsts(1, n)\n        assert len(trees) == count_bsts(n) and len({pre(t) for t in trees}) == len(trees)\n        assert all(inorder(t) == list(range(1, n + 1)) for t in trees)\n    assert sorted(t.val for t in all_bsts(3, 5)) == [3, 3, 4, 5, 5]")

### Bug hunt: Trimming too much

When a node is below the range, this trim throws away its right subtree too, though values in range may be there: this prints `[50, 60, 70]`. Return the trimmed right subtree instead, so it prints `[35, 40, 50, 60, 70]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def trim(node, lo, hi):
    if node is None:
        return None
    if node.val < lo:
        return None
    if node.val > hi:
        return trim(node.left, lo, hi)
    node.left = trim(node.left, lo, hi)
    node.right = trim(node.right, lo, hi)
    return node

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80)))
print(inorder(trim(root, 33, 75)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the right side\"\n    expect_output(\"[35, 40, 50, 60, 70]\")")

### Bug hunt: The wrong half of the second inversion

This finds swapped values from an in-order sequence, but for the second inversion it keeps the **first** value rather than the second, so it blames the wrong value: this prints `(5, 4)`. Keep the second value so it prints `(5, 2)`.

In [ ]:
def find_swapped(seq):
    first = second = None
    for prev, cur in zip(seq, seq[1:]):
        if cur < prev:
            if first is None:
                first, second = prev, cur
            else:
                second = prev
    return first, second

print(find_swapped([1, 5, 3, 4, 2, 6]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"second of the second\"\n    expect_output(\"(5, 2)\")\n    assert find_swapped([1, 2, 4, 3, 5]) == (4, 3)")

### Bug hunt: Adding up the wrong way

This "greater sum" walk goes left first, so each value becomes the sum of everything **smaller** than or equal to it: this prints `[1, 3, 6, 10]`. Walk right first so each value becomes the sum of everything greater than or equal to it, and it prints `[10, 9, 7, 4]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def greater_sum(root):
    total, stack, node = 0, [], root
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        total += node.val
        node.val = total
        node = node.right
    return root

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

T = TreeNode
print(inorder(greater_sum(T(3, T(2, T(1)), T(4)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"largest first\"\n    expect_output(\"[10, 9, 7, 4]\")")

### Puzzle: Where paths split

Arrange the lines into the BST lowest common ancestor. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Use the BST built by inserting 8, 3, 10, 1, 6, 14, 4, 7, 13 in that order.

p = ___   # the LCA of 4 and 7
q = ___   # the LCA of 1 and 13
r = ___   # the distance from 4 to 13
s = ___   # the in-order traversal after trimming to 5..13
t = ___   # how many different BSTs can hold 1, 2, 3, 4

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q == 8 and r == 6, \"p, q and r\"\n    assert s == [6, 7, 8, 10, 13] and t == 14, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the largest BST inside a tree

A binary tree that isn't a BST may still contain subtrees that are. Write `largest_bst(root)`: the number of nodes in the largest subtree (a node and **all** its descendants) that is a valid BST, with no duplicates. Do it in one O(n) post-order pass: each node reports to its parent whether its subtree is a BST, its size, and its minimum and maximum values. Then `largest_bst_sum(root)`: the largest **sum** of values over BST subtrees (0 if every BST subtree has a negative sum, since the empty tree counts).

The checks compare against a slow version on hundreds of random trees and time a 50,000-node tree. Use an explicit stack (two-stack post-order from Week 45 Day 3 works well), since random trees can be deep.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def largest_bst(root):
    ___

def largest_bst_sum(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\nT = TreeNode\ndef slow(root):\n    def ok(n, lo, hi):\n        return n is None or (lo < n.val < hi and ok(n.left, lo, n.val) and ok(n.right, n.val, hi))\n    def walk(n):\n        return [] if n is None else [n] + walk(n.left) + walk(n.right)\n    def vals(n):\n        return [] if n is None else [n.val] + vals(n.left) + vals(n.right)\n    best_size, best_sum = 0, 0\n    for n in walk(root):\n        if ok(n, float(\"-inf\"), float(\"inf\")):\n            v = vals(n)\n            best_size, best_sum = max(best_size, len(v)), max(best_sum, sum(v))\n    return best_size, best_sum\ndef rand_tree(rng, n):\n    if n == 0:\n        return None\n    k = rng.randint(0, n - 1)\n    return T(rng.randint(-20, 20), rand_tree(rng, k), rand_tree(rng, n - 1 - k))\ndef test_small():\n    \"examples\"\n    t = T(10, T(5, T(1), T(8)), T(15, None, T(7)))\n    assert largest_bst(t) == 3 and largest_bst_sum(t) == 14\n    assert largest_bst(None) == 0 and largest_bst(T(1, T(1))) == 1\n    assert largest_bst_sum(T(-5, T(-9), T(-1))) == 0\ndef test_random():\n    \"against a slow version\"\n    rng = random.Random(15)\n    for _ in range(300):\n        t = rand_tree(rng, rng.randint(0, 25))\n        assert (largest_bst(t), largest_bst_sum(t)) == slow(t)\ndef test_big():\n    \"deep and large\"\n    chain = None\n    for v in range(50_000):\n        chain = T(v, chain)\n    t0 = time.perf_counter()\n    assert largest_bst(chain) == 50_000\n    chain.left.left.val = -1\n    assert largest_bst(chain) == 49_997\n    assert time.perf_counter() - t0 < 5")

## Warm-up

While both values are on the same side of a node, their paths from the root go the same way. The first node where they don't (or where one of them is the node itself) is the **lowest common ancestor**. In a general tree that needs a full post-order search (Week 45); here it's one walk down.

## Where paths split

The **lowest common ancestor** (LCA) of two values `a` and `b` in a BST is the first node on the search path where `a` and `b` go different ways, or where one of them equals the node. Every node above it is an ancestor of both; below it, the paths diverge. Cost: O(h), touching nothing off the path.

Once you have the LCA, path questions are easy:

- The **distance** between `a` and `b` (in edges) is depth of `a` below the LCA plus depth of `b` below it.
- The **path** from `a` to `b` goes up from `a` to the LCA, then down to `b`.

This assumes both values are in the tree; checking that first is part of today's exercise.

## Reshaping a BST

**Trimming** keeps only the values in `lo..hi`. The recursive, subtree-returning style from Day 2 makes it short. At each node:

- if its value is **below** `lo`, it and its whole left subtree go; the answer is the trimmed **right** subtree;
- if it's **above** `hi`, the answer is the trimmed **left** subtree;
- otherwise it stays, with both subtrees trimmed.

No values move and the BST property is kept, because every surviving subtree was already on the correct side of every surviving ancestor.

Walking **in reverse order** (right, node, left) visits values from largest to smallest. That's what you need for "replace each value with the sum of all values greater than or equal to it", a running total carried across the walk:

## Repairing a BST

Suppose two values in a BST were **swapped** by mistake. Its in-order sequence is then sorted except for those two, and you can find them by looking for **inversions**, places where a value is smaller than the one before it:

- If the swapped values were **next to each other** in sorted order, there's one inversion: swap its two values back. In `1 2 4 3 5`, the inversion is (4, 3).
- Otherwise there are two: the culprits are the **first** value of the first inversion and the **second** value of the second. In `1 5 3 4 2 6`, the inversions are (5, 3) and (4, 2), and the swapped values are 5 and 2.

To repair the tree itself, do the same during an in-order walk, remembering the **nodes**, not just the values, then swap their values. It's O(n) time and O(h) extra space, with no list of all the values needed.

## Two trees at once

**Merging** two BSTs into one sorted list could put all the values into a list and sort it, O((m + n) log(m + n)). Better: an in-order iterator over each tree (Week 45 Day 3) gives two sorted streams, which merge exactly like Week 19's merge step, in O(m + n) time and O(h₁ + h₂) extra space beyond the output.

(Using `None` as "finished" is fine here because the trees hold numbers; a sentinel object would be safer in general.) The merged list is sorted, so Day 1's `from_sorted` turns it straight into a balanced BST holding both trees' values.

## Counting shapes

How many different BSTs can hold the values 1 to n? Choose the root `r`: the values below it (r − 1 of them) form the left subtree in any of its possible shapes, and the n − r above it form the right. So

count(n) = sum over r from 1 to n of count(r − 1) × count(n − r), with count(0) = 1.

That's 1, 1, 2, 5, 14, 42, 132, ... the **Catalan numbers**, named after Eugène Catalan (1838), which also count balanced bracket strings and many other things. Memoising (Week 16) or a bottom-up table makes it O(n²):

Nearly 1.8 billion shapes for 19 values, and only a handful of them have height 4. That's another way to see why insertion order matters so much.

## Practice

## Quiz

## Daily challenge

## Before you go

Take a sorted list of seven numbers, swap two of them, and find the inversions by hand. Then do the same with two neighbours. Tomorrow: saving trees to text and loading them back, going further than last week's `#` markers.